# Exercise 1: Hydrogenation of bimetallic nanoparticles

## Background

The hydrogen storage capacity of metallic nanoparticles depends on three key
structural parameters: **size**, **shape**, and **chemical composition**.
For AuPd bimetallic systems, even small changes in the Au:Pd ratio or in the
degree of chemical ordering can drastically change how much hydrogen the particle
absorbs at a given temperature and pressure.

In this exercise we use **grand-canonical Monte Carlo (GCMC)** to simulate
hydrogen adsorption on AuPd nanoparticles. The chemical potential $\mu$ of the
hydrogen reservoir controls the H loading, and by sweeping $\mu$ we can build
the adsorption isotherm $\langle N_H \rangle(\mu)$.

The interatomic interactions are described by a **GAP machine-learning
potential** trained on DFT data for the Au–Pd–H system, which gives
near-DFT accuracy at a fraction of the cost.

## Goals

1. Generate an AuPd nanoparticle using a realistic "cooking" recipe:
   anneal at high temperature → quench → relax → anneal.
2. Run GCMC to hydrogenate the cooked nanoparticle and monitor H uptake.
3. Compare your cooked structure against two reference structures:
   an ordered and a disordered truncated octahedron.
4. Analyse energy and H-concentration convergence.
5. Explore the effect of GCMC parameters on the result.
6. *(Optional)* Repeat for a larger nanoparticle ($N = 79$ atoms).

## mc.log column reference

The `mc.log` file written by TurboGAP has the following columns:

| Column | Quantity |
|--------|----------|
| 0 | MC step |
| 1 | mc_move |
| 2 | Accepted |
| 3 | E_trial (eV) |
| 4 | E_current (eV) |
| 5 | Total number of atoms |
| 6 | Species |
| 7 | Number of H atoms |


---
## Background — Nanoparticle stability and structure–property relationships

### 1. Why nanoparticles behave differently from bulk

A bulk metal crystal has a well-defined, repeating structure where the vast
majority of atoms sit in the interior. As the particle shrinks, an increasing
fraction of atoms sits at the **surface**, at **edges**, or at **corners** —
sites that are geometrically and electronically very different from interior bulk
sites.

The fraction of surface atoms $f_s$ scales roughly as:

$$f_s \approx \frac{N_s}{N_{\mathrm{tot}}} \propto N_{\mathrm{tot}}^{-1/3}
\propto \frac{1}{d}$$

where $d$ is the particle diameter. For a 1 nm particle, nearly all atoms are
at the surface; for a 10 nm particle, only ~10% are. This has direct
consequences for:

- **Thermodynamic stability** — surface atoms have fewer neighbours and higher
  energy, so surface energy $\gamma$ dominates the total free energy at small
  sizes.
- **Melting point depression** — the Gibbs–Thomson relation predicts that the
  melting temperature decreases as $\sim 1/d$, allowing nanoparticles to be
  "cooked" at temperatures that would leave bulk metals solid.
- **Catalytic and adsorption properties** — active sites are predominantly on
  the surface, so surface area per unit mass increases as $\sim 1/d$, and the
  *type* of surface site (terrace, edge, corner) changes with size and shape.

### 2. Magic sizes and geometric shells

Nanoparticles do not adopt arbitrary sizes — they are most stable at specific
"**magic**" atom counts corresponding to complete geometric shells. For FCC
metals the most common closed-shell polyhedra are:

| Shape | Magic sizes $N$ |
|-------|----------------|
| Cuboctahedron | 13, 55, 147, 309, 561 |
| Truncated octahedron (TO) | 38, 79, 201, 405 |
| Icosahedron | 13, 55, 147, 309 |

In this exercise we work with $N = 55$ and $N = 79$ atoms — both magic sizes.
The **truncated octahedron** (Figure below) exposes (111) and (100) facets,
which are the lowest-energy surfaces of FCC metals and are therefore particularly
stable. The (111) facets appear as hexagonal faces and the (100) facets as
square faces.

The cohesive energy per atom $E_c / N$ increases (becomes more negative) with
particle size as more atoms occupy high-coordination bulk-like sites:

$$\frac{E_c}{N}(\text{NP}) \approx \frac{E_c}{N}(\text{bulk})
\left(1 - \frac{c}{N^{1/3}}\right)$$

where $c$ is a shape-dependent constant. This is why larger particles are
thermodynamically more stable — but also less reactive.

### 3. Chemical ordering in bimetallic nanoparticles

For a bimetallic AuPd particle, beyond size and shape there is a third degree
of freedom: **how the two elements are distributed**. Three limiting cases exist:

| Ordering | Description | Tendency |
|----------|-------------|---------|
| **Random alloy** | Au and Pd occupy sites with probability $x_{\mathrm{Au}}$ | Entropic stabilisation at high $T$ |
| **Core–shell** | One element segregates to the surface | Driven by surface energy difference |
| **Ordered intermetallic** | Au and Pd alternate on a superlattice | Enthalpic stabilisation at low $T$ |

In Au–Pd systems, **Au tends to segregate to the surface** because it has a
lower surface energy than Pd and a larger atomic radius. At low temperatures
this produces a Pd-rich core / Au-rich shell structure. At high temperatures
configurational entropy favours mixing, explaining why the "melt + quench"
protocol in Part 1 produces a disordered state.

The degree of chemical ordering has direct consequences for catalysis and
hydrogen uptake, as we will see in Part 2.

### 4. Hydrogen adsorption sites and coordination

Hydrogen does not adsorb equally everywhere on a metal surface.
The binding energy depends strongly on the **local coordination environment**
of the adsorption site:

| Site type | Description | Coordination | Typical $E_b$ trend |
|-----------|-------------|-------------|---------------------|
| **Top** | Atop a single metal atom | 1 | Weakest |
| **Bridge** | Between two metal atoms | 2 | Intermediate |
| **Hollow (fcc)** | In a 3-fold hollow, no atom below | 3 | Strong |
| **Hollow (hcp)** | In a 3-fold hollow, atom below | 3 | Strong |

On a (111) facet of an FCC metal, the fcc hollow is usually the most stable
adsorption site. On (100) facets, the 4-fold hollow is preferred.
At **edge** and **corner** sites the coordination is lower and the d-band
is narrower and shifted — in Pd-rich environments this typically *strengthens*
H binding relative to flat terraces.

For AuPd alloys, the local chemical environment matters too.
A Pd site surrounded by Au neighbours has a different d-band centre than a
Pd site in a pure-Pd region, leading to a distribution of binding energies
across the particle — this is sometimes called the **ligand effect**.

The **ensemble effect** is also important: certain reactions (including H$_2$
dissociation) require a specific arrangement of neighbouring atoms.
A Pd atom completely surrounded by Au (an isolated Pd site) may not dissociate
H$_2$ at all, while a Pd ensemble of three or more contiguous atoms can.

### 5. Connecting structure to GCMC output

In the GCMC simulation:

- The **chemical potential** $\mu$ maps directly to the H$_2$ partial pressure
  via $\mu = \frac{1}{2}\mu_{\mathrm{H_2}}$, allowing comparison with
  experiment.
- The **equilibrium H concentration** $\langle N_H \rangle / N_{\mathrm{metal}}$
  at a given $\mu$ reflects the thermally averaged occupation of all
  accessible binding sites.
- Sites with stronger binding ($E_b \ll 0$) fill first (at low $\mu$);
  weaker sites fill later as $\mu$ increases.
- Differences between ordered and disordered particles (GCMC-2 vs GCMC-3)
  directly reflect differences in the **distribution of binding site
  energies** caused by changes in local chemical environment.

> **Key question to keep in mind:** when you compare ordered and disordered
> particles in Part 2, ask yourself which effect dominates —
> the ensemble effect (availability of Pd ensembles for H$_2$ dissociation)
> or the ligand effect (change in binding energy at individual Pd sites)?

## Setup

Unpack the GAP potential files and import the required Python packages.
Run the cell below once at the start of the session.


In [1]:
import subprocess
subprocess.run(["tar", "-xzf", "gap_files.tar.gz"])

from ase.io import write, read
from weas_widget import WeasWidget
import numpy as np
import matplotlib.pyplot as plt


tar (child): gap_files.tar.xz: Cannot open: No such file or directory
tar (child): Error is not recoverable: exiting now
tar: Child returned status 2
tar: Error is not recoverable: exiting now


ModuleNotFoundError: No module named 'weas_widget'

---
## Part 1 — Nanoparticle preparation ("cooking")

A realistic nanoparticle is not a perfect crystal fragment.
We mimic experimental synthesis by running a four-step thermal protocol:

| Step | Script | Purpose |
|------|--------|---------|
| 1. Initial structure | `make_NP.py` | Place 55 Au/Pd atoms in a sphere |
| 2. Anneal at high temperature | `1.anneal.sh` | MD at 800 K — randomise atom positions and chemical order |
| 3. Quench | `2.quench.sh` | Rapid cooling to 100 K — freeze in a low-energy disordered state |
| 4. Relax | `3.relax.sh` | Energy minimisation — remove residual forces |
| 5. Anneal | `4.anneal2.sh` | MD at 600 K — allow local rearrangements towards equilibrium |

Each step reads the output of the previous one.
Visualise the structure after each step to see how it evolves.


### Step 1 — Initial structure

Run `python make_NP.py` in the terminal to generate the starting nanoparticle. The script places Au and Pd atoms on an FCC lattice in a 1:1 ratio, randomly distributed with roughly spherical shape.


In [ ]:
atoms = read("structures/atoms0.xyz")
print(f"Composition: {atoms.get_chemical_formula()}")
print(f"Number of atoms: {len(atoms)}")

viewer = WeasWidget()
viewer.avr.model_style = 0
viewer.from_ase(atoms)
viewer


### Step 2 — Anneal at high temperature (800 K)

Copy and run the melt script:
```bash
cp scripts/1.anneal.sh .
bash 1.anneal.sh
```
The nanoparticle is heated to 800 K, below the melting temperature to favor crystallisation 


In [ ]:
atoms = read("structures/atoms1.xyz")
print(f"Energy after melt: {atoms.get_potential_energy():.3f} eV")

viewer = WeasWidget()
viewer.avr.model_style = 0
viewer.from_ase(atoms)
viewer


### Step 3 — Quench (800 K → 100 K)

```bash
cp scripts/2.quench.sh .
bash 2.quench.sh
```
The particle is rapidly cooled. The disordered atomic arrangement from the melt
is preserved — this mimics a kinetically trapped, chemically disordered state.


In [ ]:
atoms = read("structures/atoms2.xyz")
print(f"Energy after quench: {atoms.get_potential_energy():.3f} eV")

viewer = WeasWidget()
viewer.avr.model_style = 0
viewer.from_ase(atoms)
viewer


### Step 4 — Relax (energy minimisation)

```bash
cp scripts/3.relax.sh .
bash 3.relax.sh
```
A geometry optimisation removes residual forces from the MD.
The structure should have a lower energy than after the quench.

> **Question:** How does the energy per atom compare with your neighbours?
> A lower energy means a more stable chemical ordering was found during the melt.


In [ ]:
atoms = read("structures/atoms3.xyz")
E_relax = atoms.get_potential_energy()
N_atoms = len(atoms)
print(f"Energy after relax:          {E_relax:.3f} eV")
print(f"Energy per atom:             {E_relax/N_atoms:.4f} eV/atom")

viewer = WeasWidget()
viewer.avr.model_style = 0
viewer.from_ase(atoms)
viewer


### Step 5 — Anneal (600 K)

```bash
cp scripts/4.anneal2.sh .
bash 4.anneal2.sh
```
A second, MD annealing run allows the particle to explore nearby
configurations and settle into a more stable structure.
This is the final "cooked" nanoparticle you will use for GCMC.


In [ ]:
atoms = read("structures/atoms4.xyz")
E_anneal = atoms.get_potential_energy()
print(f"Energy after anneal:         {E_anneal:.3f} eV")
print(f"Energy per atom:             {E_anneal/N_atoms:.4f} eV/atom")
print(f"Energy gain from anneal:     {E_relax - E_anneal:.3f} eV")

viewer = WeasWidget()
viewer.avr.model_style = 0
viewer.from_ase(atoms)
viewer


---
## Part 2 — Grand-canonical Monte Carlo (GCMC)

We now run GCMC to insert hydrogen into the nanoparticle.
The simulation operates at fixed $\mu$, $V$, $T$ — the $\mu$VT ensemble.
At each MC step, TurboGAP randomly attempts one of:

- **Displacement** of a randomly chosen atom
- **Insertion** of a H atom at a random position
- **Removal** of a randomly chosen H atom

The acceptance probabilities follow the Metropolis criterion derived in the
lectures. The chemical potential $\mu$ controls the H reservoir — a higher
$\mu$ favours insertion and leads to a higher H concentration.

### GCMC-1 — your cooked nanoparticle


Copy and run the first GCMC script:
```bash
cp scripts/5.gcmc-1.sh .
bash 5.gcmc-1.sh
```
This runs GCMC on the annealed structure from Part 1.
Visualise the trajectory to see H atoms being inserted and removed.


In [ ]:
atoms = read("5.gcmc-1/mc_all.xyz", index=':')
print(f"Number of saved MC frames: {len(atoms)}")
print(f"H atoms in last frame:     "
      f"{atoms[-1].get_chemical_symbols().count('H')}")

viewer = WeasWidget()
viewer.avr.model_style = 0
viewer.from_ase(atoms)
viewer


Plot the energy and H concentration as a function of MC step.
Check that both quantities have converged — if they are still drifting,
the simulation has not yet equilibrated.


In [ ]:
N = len(read("structures/atoms4.xyz"))-2  # number of metal atoms

data   = np.genfromtxt('5.gcmc-1/mc.log')
nsteps = data[:, 0]
ener   = data[:, 4]
cH     = data[:, 7]

fig, axs = plt.subplots(1, 2, figsize=(8, 4), layout='constrained')

axs[0].plot(nsteps, ener, color='steelblue', lw=0.8)
axs[0].set_xlabel('MC steps')
axs[0].set_ylabel('Energy (eV)')
axs[0].set_title('Total energy')

axs[1].plot(nsteps, cH / N * 100, color='darkorange', lw=0.8)
axs[1].set_xlabel('MC steps')
axs[1].set_ylabel('H / metal (%)')
axs[1].set_title('H concentration')

# Equilibrium estimate (last 50% of run)
half = len(cH) // 2
print(f"Mean H concentration (last 50%): {cH[half:].mean()/N*100:.1f} %")

plt.show()


### GCMC-2 — ordered cuboctahedron

We now compare against a reference structure: a **perfectly ordered** AuPd
cuboctahedron with 55 atoms (`Cu55_AuPd.xyz`).
In this structure Au and Pd occupy well-defined sublattice sites.

```bash
cp scripts/6.gcmc-2.sh .
bash 6.gcmc-2.sh
```


In [ ]:
atoms = read("6.gcmc-2/mc_all.xyz", index=':')
print(f"Number of saved MC frames: {len(atoms)}")
print(f"H atoms in last frame:     "
      f"{atoms[-1].get_chemical_symbols().count('H')}")

viewer = WeasWidget()
viewer.avr.model_style = 0
viewer.from_ase(atoms)
viewer


In [ ]:
N = 55

data   = np.genfromtxt('6.gcmc-2/mc.log')
nsteps = data[:, 0]
ener   = data[:, 4]
cH     = data[:, 7]

fig, axs = plt.subplots(1, 2, figsize=(8, 4), layout='constrained')

axs[0].plot(nsteps, ener, color='steelblue', lw=0.8)
axs[0].set_xlabel('MC steps')
axs[0].set_ylabel('Energy (eV)')
axs[0].set_title('Total energy — ordered NP')

axs[1].plot(nsteps, cH / N * 100, color='darkorange', lw=0.8)
axs[1].set_xlabel('MC steps')
axs[1].set_ylabel('H / metal (%)')
axs[1].set_title('H concentration — ordered NP')

half = len(cH) // 2
print(f"Mean H concentration (last 50%): {cH[half:].mean()/N*100:.1f} %")

plt.show()


### GCMC-3 — disordered truncated octahedron

Finally, compare against a **chemically disordered** truncated octahedron
(`Cu55_dis_AuPd.xyz`) with the same size and shape but randomised
Au/Pd occupancy.

```bash
cp scripts/7.gcmc-3.sh .
bash 7.gcmc-3.sh
```

> **Question:** Does chemical ordering matter for hydrogen uptake?
> Compare the mean H concentrations from GCMC-2 and GCMC-3.
> What does this tell you about the role of local chemical environment?


In [ ]:
atoms = read("7.gcmc-3/mc_all.xyz", index=':')
print(f"Number of saved MC frames: {len(atoms)}")
print(f"H atoms in last frame:     "
      f"{atoms[-1].get_chemical_symbols().count('H')}")

viewer = WeasWidget()
viewer.avr.model_style = 0
viewer.from_ase(atoms)
viewer


In [ ]:
N = 79

fig, axs = plt.subplots(1, 2, figsize=(8, 4), layout='constrained')
labels = ['ordered', 'disordered']
colors = ['steelblue', 'darkorange']

for idx, (folder, label, color) in enumerate(zip(
        ['6.gcmc-2', '7.gcmc-3'], labels, colors)):
    data   = np.genfromtxt(f'{folder}/mc.log')
    nsteps = data[:, 0]
    ener   = data[:, 4]
    cH     = data[:, 7]

    axs[0].plot(nsteps, ener, color=color, lw=0.8, label=label)
    axs[1].plot(nsteps, cH / N * 100, color=color, lw=0.8, label=label)

    half = len(cH) // 2
    print(f"{label:12s}  mean H conc. (last 50%): "
          f"{cH[half:].mean()/N*100:.1f} %")

axs[0].set_xlabel('MC steps')
axs[0].set_ylabel('Energy (eV)')
axs[0].set_title('Total energy')
axs[0].legend()

axs[1].set_xlabel('MC steps')
axs[1].set_ylabel('H / metal (%)')
axs[1].set_title('H concentration')
axs[1].legend()

plt.show()


In [ ]:
"""
Analyse and plot hydrogen adsorption on a PdAu nanoparticle.
Coordination-based site classification — works for any nanoparticle shape.
"""

import re
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import Counter, defaultdict
from ase.io import read
from ase.neighborlist import NeighborList

# -----------------------------------------------------------------------
# HELPER
# -----------------------------------------------------------------------
def normalize_comp(comp):
    return ''.join(sorted(re.findall(r'[A-Z][a-z]?', comp)))

def comp_color(c):
    elems = re.findall(r'[A-Z][a-z]?', c)
    if all(e == 'Pd' for e in elems):
        return 'steelblue'
    if all(e == 'Au' for e in elems):
        return 'goldenrod'
    return 'mediumseagreen'

print(f"Working directory: {os.getcwd()}")

# -----------------------------------------------------------------------
# 1. LOAD STRUCTURE
# -----------------------------------------------------------------------
atoms = read('mc_current.xyz')
atoms.pbc = False

syms     = atoms.get_chemical_symbols()
n_atoms  = len(atoms)
h_idx    = [i for i, s in enumerate(syms) if s == 'H']
pd_idx   = [i for i, s in enumerate(syms) if s == 'Pd']
au_idx   = [i for i, s in enumerate(syms) if s == 'Au']
metal_idx = pd_idx + au_idx

print(f"Pd atoms : {len(pd_idx)}")
print(f"Au atoms : {len(au_idx)}")
print(f"H atoms  : {len(h_idx)}")

# -----------------------------------------------------------------------
# 2. BUILD NEIGHBOR LIST
# -----------------------------------------------------------------------
cutoff_H_metal  = 2.5   # H-metal bond cutoff (Angstrom)
cutoff_metal    = 3.5   # metal-metal cutoff for coordination number

# use max cutoff for the neighbor list
nl = NeighborList(
    [max(cutoff_H_metal, cutoff_metal) / 2] * n_atoms,
    self_interaction=False,
    bothways=True,
)
nl.update(atoms)

# -----------------------------------------------------------------------
# 3. COORDINATION NUMBER — identify surface vs bulk metal atoms
# -----------------------------------------------------------------------
coord_numbers = np.zeros(n_atoms, dtype=int)
for i in metal_idx:
    neighbors, _ = nl.get_neighbors(i)
    # count only metal neighbors within metal cutoff
    metal_neigh = [n for n in neighbors
                   if syms[n] in ('Pd', 'Au')
                   and np.linalg.norm(atoms.positions[i] -
                                      atoms.positions[n]) < cutoff_metal]
    coord_numbers[i] = len(metal_neigh)

surf_idx = [i for i in metal_idx if coord_numbers[i] < 11]
bulk_idx = [i for i in metal_idx if coord_numbers[i] >= 11]

print(f"\nSurface metal atoms : {len(surf_idx)}")
print(f"Bulk metal atoms    : {len(bulk_idx)}")
print(f"Mean CN surface     : {np.mean([coord_numbers[i] for i in surf_idx]):.2f}")
print(f"Mean CN bulk        : {np.mean([coord_numbers[i] for i in bulk_idx]):.2f}")

# CN distribution
cn_counter = Counter(coord_numbers[i] for i in metal_idx)
print("\nCN distribution (metal atoms):")
for cn, count in sorted(cn_counter.items()):
    print(f"  CN={cn:2d}: {count}")

# -----------------------------------------------------------------------
# 4. CLASSIFY H ATOMS BY LOCAL COORDINATION
# -----------------------------------------------------------------------
site_data = []

for hi in h_idx:
    neighbors, _ = nl.get_neighbors(hi)

    # metal neighbors within H-metal cutoff
    metal_neigh = [n for n in neighbors
                   if syms[n] in ('Pd', 'Au')
                   and np.linalg.norm(atoms.positions[hi] -
                                      atoms.positions[n]) < cutoff_H_metal]
    n_metal = len(metal_neigh)
    comp_counter = Counter(syms[n] for n in metal_neigh)
    comp_str = ''.join(sorted([syms[n] for n in metal_neigh]))

    # site classification by number of metal neighbors
    if n_metal == 0:
        site = 'unbound'
    elif n_metal == 1:
        site = 'ontop'
    elif n_metal == 2:
        site = 'bridge'
    elif n_metal == 3:
        site = 'hollow3'
    elif n_metal == 4:
        site = 'hollow4'
    else:
        site = 'subsurface'

    # check if H is subsurface: all metal neighbors are bulk atoms
    if n_metal > 0 and all(coord_numbers[n] >= 11 for n in metal_neigh):
        site = 'subsurface'

    # bond lengths to each metal neighbor
    bond_lengths = [np.linalg.norm(atoms.positions[hi] -
                                   atoms.positions[n])
                    for n in metal_neigh]
    mean_bond = np.mean(bond_lengths) if bond_lengths else 0.0

    site_data.append({
        'h_index':   hi,
        'site':      site,
        'n_metal':   n_metal,
        'n_Pd':      comp_counter.get('Pd', 0),
        'n_Au':      comp_counter.get('Au', 0),
        'comp':      normalize_comp(comp_str) if comp_str else 'none',
        'mean_bond': mean_bond,
        'position':  atoms.positions[hi],
    })

# -----------------------------------------------------------------------
# 5. PRINT SUMMARY
# -----------------------------------------------------------------------
by_site = Counter(d['site'] for d in site_data)
by_comp = Counter(d['comp'] for d in site_data)

print(f"\n--- H site classification (coordination-based) ---")
for k, v in sorted(by_site.items()):
    print(f"  {k:15s}: {v}")

print(f"\n--- H site compositions ---")
for k, v in sorted(by_comp.items()):
    print(f"  {k:20s}: {v}")

print(f"\nMean metal neighbors per H : "
      f"{np.mean([d['n_metal'] for d in site_data]):.2f}")
print(f"Mean Pd neighbors per H    : "
      f"{np.mean([d['n_Pd'] for d in site_data]):.2f}")
print(f"Mean Au neighbors per H    : "
      f"{np.mean([d['n_Au'] for d in site_data]):.2f}")
print(f"Mean H-metal bond length   : "
      f"{np.mean([d['mean_bond'] for d in site_data if d['mean_bond'] > 0]):.3f} Å")

# -----------------------------------------------------------------------
# 6. PLOT 1 — bar charts: site type and composition
# -----------------------------------------------------------------------
site_colors_map = {
    'ontop':      'orchid',
    'bridge':     'coral',
    'hollow3':    'steelblue',
    'hollow4':    'goldenrod',
    'subsurface': 'gray',
    'unbound':    'black',
}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# site type
ax = axes[0]
sites  = sorted(by_site.keys())
counts = [by_site[s] for s in sites]
colors = [site_colors_map.get(s, 'gray') for s in sites]
bars   = ax.bar(sites, counts, color=colors, alpha=0.85, edgecolor='white')
ax.bar_label(bars, fontsize=10)
ax.set_ylabel('Number of H atoms')
ax.set_title('H site types (coordination-based)')
ax.set_xticklabels(sites, rotation=30, ha='right')
patches = [mpatches.Patch(color=c, label=s)
           for s, c in site_colors_map.items() if s in by_site]
ax.legend(handles=patches, fontsize=8)

# composition
ax = axes[1]
comps  = sorted(by_comp.keys())
counts = [by_comp[c] for c in comps]
colors = [comp_color(c) for c in comps]
bars   = ax.bar(comps, counts, color=colors, alpha=0.85, edgecolor='white')
ax.bar_label(bars, fontsize=10)
ax.set_ylabel('Number of H atoms')
ax.set_title('H site compositions')
ax.set_xticks(range(len(comps)))
ax.set_xticklabels(comps, rotation=30, ha='right')
patches = [
    mpatches.Patch(color='steelblue',     label='pure Pd'),
    mpatches.Patch(color='mediumseagreen', label='mixed PdAu'),
    mpatches.Patch(color='goldenrod',      label='pure Au'),
]
ax.legend(handles=patches, fontsize=8)

plt.suptitle('H adsorption on PdAu nanoparticle', fontsize=13)
plt.tight_layout()
plt.savefig('site_analysis.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved: site_analysis.png")

# -----------------------------------------------------------------------
# 7. PLOT 2 — heatmap: site type vs composition
# -----------------------------------------------------------------------
sites_list = sorted(set(d['site'] for d in site_data))
comps_list = sorted(set(d['comp'] for d in site_data))

mat = np.zeros((len(sites_list), len(comps_list)), dtype=int)
for d in site_data:
    if d['site'] in sites_list and d['comp'] in comps_list:
        mat[sites_list.index(d['site'])][comps_list.index(d['comp'])] += 1

fig, ax = plt.subplots(figsize=(10, 5))
im = ax.imshow(mat, cmap='Blues', aspect='auto')
ax.set_xticks(range(len(comps_list)))
ax.set_yticks(range(len(sites_list)))
ax.set_xticklabels(comps_list, rotation=30, ha='right', fontsize=10)
ax.set_yticklabels(sites_list, fontsize=10)
ax.set_xlabel('Composition')
ax.set_ylabel('Site type')
ax.set_title('H site type vs composition — PdAu nanoparticle')
for i in range(len(sites_list)):
    for j in range(len(comps_list)):
        if mat[i, j] > 0:
            ax.text(j, i, str(mat[i, j]),
                    ha='center', va='center', fontsize=10,
                    color='white' if mat[i, j] > mat.max() * 0.5 else 'black')
plt.colorbar(im, ax=ax, label='count')
plt.tight_layout()
plt.savefig('heatmap_site_comp.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved: heatmap_site_comp.png")

# -----------------------------------------------------------------------
# 8. PLOT 4 — coordination number distribution of metal atoms
# -----------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# CN distribution for Pd vs Au
ax = axes[0]
for elem, idx_list, color in [('Pd', pd_idx, 'steelblue'),
                                ('Au', au_idx, 'goldenrod')]:
    cns = [coord_numbers[i] for i in idx_list]
    if cns:
        cn_vals = sorted(set(cns))
        counts  = [cns.count(c) for c in cn_vals]
        ax.plot(cn_vals, counts, 'o-', color=color,
                label=elem, linewidth=2, markersize=6)
ax.axvline(11, color='red', linestyle='--', linewidth=1,
           label='surface/bulk threshold (CN=11)')
ax.set_xlabel('Coordination number')
ax.set_ylabel('Count')
ax.set_title('CN distribution: Pd vs Au')
ax.legend()

# CN of metal atoms bonded to H
ax = axes[1]
h_bonded_cn = []
h_bonded_elem = []
for d in site_data:
    hi = d['h_index']
    neighbors, _ = nl.get_neighbors(hi)
    metal_neigh = [n for n in neighbors
                   if syms[n] in ('Pd', 'Au')
                   and np.linalg.norm(atoms.positions[hi] -
                                      atoms.positions[n]) < cutoff_H_metal]
    for n in metal_neigh:
        h_bonded_cn.append(coord_numbers[n])
        h_bonded_elem.append(syms[n])

for elem, color in [('Pd', 'steelblue'), ('Au', 'goldenrod')]:
    cns = [cn for cn, el in zip(h_bonded_cn, h_bonded_elem) if el == elem]
    if cns:
        cn_vals = sorted(set(cns))
        counts  = [cns.count(c) for c in cn_vals]
        ax.plot(cn_vals, counts, 'o-', color=color,
                label=elem, linewidth=2, markersize=6)
ax.axvline(11, color='red', linestyle='--', linewidth=1,
           label='surface/bulk threshold (CN=11)')
ax.set_xlabel('Coordination number of metal atom bonded to H')
ax.set_ylabel('Count')
ax.set_title('CN of metal atoms bonded to H')
ax.legend()

plt.suptitle('Coordination numbers — PdAu nanoparticle', fontsize=13)
plt.tight_layout()
plt.savefig('coordination.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved: coordination.png")

# -----------------------------------------------------------------------
# 9. WRITE OVITO XYZ WITH SITE TYPE COLUMN
# -----------------------------------------------------------------------
site_label = {
    'ontop':      1,
    'bridge':     2,
    'hollow3':    3,
    'hollow4':    4,
    'subsurface': 5,
    'unbound':    6,
}

h_site_type = {d['h_index']: site_label.get(d['site'], 0)
               for d in site_data}

with open('mc_current.xyz', 'r') as f:
    lines = f.readlines()

n_file    = int(lines[0].strip())
comment   = lines[1].strip()
datalines = lines[2:2 + n_file]

site_type_col = np.zeros(n_file, dtype=int)
for i, atom in enumerate(atoms):
    if atom.symbol == 'H':
        site_type_col[i] = h_site_type.get(i, 0)

with open('nanoparticle_H_colored.xyz', 'w') as f:
    f.write(f"{n_file}\n")
    f.write(f"{comment} "
            f"site_type=0:metal,1:ontop,2:bridge,3:hollow3,4:hollow4,"
            f"5:subsurface,6:unbound\n")
    for i, line in enumerate(datalines):
        f.write(f"{line.rstrip()} {site_type_col[i]}\n")

print("Saved: nanoparticle_H_colored.xyz")
print("\nDone. Figures: site_analysis.png, heatmap_site_comp.png, "
      "bond_lengths.png, coordination.png")

---
## Part 3 — Parameter sensitivity

Explore how GCMC parameters affect the result.
For each experiment, modify the relevant parameter in the run script,
re-run, and plot the H concentration trace.

| Parameter | TurboGAP keyword | Suggested values to try |
|-----------|-----------------|--------------------------|
| Max displacement | `mc_move_max` | 0.05, 0.20, 0.80 Å |
| Temperature | `temperature` | 200, 300, 500 K |
| Move ratio (ins:disp) | `mc_moves` | insertion-only vs mixed |
| Chemical potential | `mu` | vary by ±0.2 eV |

> **Question:** Which parameter has the largest effect on the equilibrium
> H concentration? Which affects only the convergence speed?


In [ ]:
# Template for parameter sensitivity analysis.
# Modify 'folders' and 'labels' to match your run directories.

N = 79
folders = ['5.gcmc-1']   # add your new run directories here
labels  = ['baseline']   # add matching labels here

fig, axs = plt.subplots(1, 2, figsize=(8, 4), layout='constrained')

for folder, label in zip(folders, labels):
    data   = np.genfromtxt(f'{folder}/mc.log')
    nsteps = data[:, 0]
    ener   = data[:, 4]
    cH     = data[:, 7]

    axs[0].plot(nsteps, ener, lw=0.8, label=label)
    axs[1].plot(nsteps, cH / N * 100, lw=0.8, label=label)

    half = len(cH) // 2
    print(f"{label:20s}  mean H conc.: {cH[half:].mean()/N*100:.1f} %")

axs[0].set_xlabel('MC steps'); axs[0].set_ylabel('Energy (eV)')
axs[0].set_title('Energy convergence'); axs[0].legend()
axs[1].set_xlabel('MC steps'); axs[1].set_ylabel('H / metal (%)')
axs[1].set_title('H concentration'); axs[1].legend()
plt.show()
